1. Importing Necessary Libraries

In [1]:
import os
import pandas as pd
import shutil
from PIL import Image
from tqdm import tqdm

2. Create YOLOv8 Dataset Folder Structure

In [2]:
# Create YOLO-friendly folder tree: yolo_dataset/{images,labels}/{train,val,test}.

output_root = "../../datasets/yolo_dataset"

# Create train val and test splits.
splits = ['train', 'val', 'test']

# Create the required directory structure
for split in splits:
    # images/<split> holds the jpg/png frames
    images_out_dir = os.path.join(output_root, 'images', split)
    # labels/<split> holds the .txt files in YOLO format: <class> <xc> <yc> <w> <h> (normalized)
    labels_out_dir = os.path.join(output_root, 'labels', split)

    os.makedirs(images_out_dir, exist_ok=True)
    os.makedirs(labels_out_dir, exist_ok=True)

print(f"Created folders under '{output_root}': {', '.join(splits)}")

Created folders under '../../datasets/yolo_dataset': train, val, test


3. Load Data

In [3]:
# dataset path
DATA_PATH = "../../annotations"
os.listdir(DATA_PATH)
# Load the split metadata (train/val/test and annotations)
# The file uses ';' as the delimiter.
df = pd.read_csv(os.path.join(DATA_PATH,'lisa_dataset_split.csv'), sep=';')

In [4]:
# Shape: (rows, cols)
df.shape

(51826, 10)

In [5]:
# Preview first 5 rows
df.head()

,image_id,label,x_min,y_min,x_max,y_max,frame,isNight,clipNames,split
0,../../datasets/lisa-traffic-light-dataset\dayT...,1,698,333,710,358,0,0,dayClip1,train
1,../../datasets/lisa-traffic-light-dataset\dayT...,1,846,391,858,411,0,0,dayClip1,train
2,../../datasets/lisa-traffic-light-dataset\dayT...,1,698,337,710,357,1,0,dayClip1,train
3,../../datasets/lisa-traffic-light-dataset\dayT...,1,847,390,859,410,1,0,dayClip1,train
4,../../datasets/lisa-traffic-light-dataset\dayT...,1,698,331,710,356,2,0,dayClip1,train


In [6]:
# Show how many samples exist in each split (train/val/test)
# Useful to sanity-check class imbalance between splits.
for split_name, count in df['split'].value_counts().items():
    print(f"{split_name}: {count} records")

train: 30707 records
test: 14433 records
val: 6686 records


In [7]:
# Count how many UNIQUE images exist in each split (train/val/test).
# This guards against duplicates: multiple rows can reference the same image_id.
unique_counts = df.groupby('split')['image_id'].nunique()

for split_name, count in unique_counts.items():
    print(f"{split_name}: {count} unique images")

test: 4603 unique images
train: 10426 unique images
val: 2984 unique images


In [8]:
# Convert labels from {1,2,3} to {0,1,2} for YOLO/ML compatibility.
# (Keep dtype integer to avoid issues downstream.)
df['label'] = df['label'] - 1

In [9]:
# Preview first 5 rows
df.head()

,image_id,label,x_min,y_min,x_max,y_max,frame,isNight,clipNames,split
0,../../datasets/lisa-traffic-light-dataset\dayT...,0,698,333,710,358,0,0,dayClip1,train
1,../../datasets/lisa-traffic-light-dataset\dayT...,0,846,391,858,411,0,0,dayClip1,train
2,../../datasets/lisa-traffic-light-dataset\dayT...,0,698,337,710,357,1,0,dayClip1,train
3,../../datasets/lisa-traffic-light-dataset\dayT...,0,847,390,859,410,1,0,dayClip1,train
4,../../datasets/lisa-traffic-light-dataset\dayT...,0,698,331,710,356,2,0,dayClip1,train


4. Export dataset to YOLO format (images/labels per split)

In [10]:
# Create YOLO directory structure for all splits you plan to use
output_root = "../../datasets/yolo_dataset"
for split in ['train', 'val', 'test']:
    os.makedirs(os.path.join(output_root, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(output_root, 'labels', split), exist_ok=True)

# Assumptions about df columns:
# df has: ['image_id','x_min','y_min','x_max','y_max','label','split']
# where split ∈ {'train','val','test'} and 'label' is 0-based (0..K-1)

# Iterate rows and write YOLO files
for _, row in tqdm(df.iterrows(), total=len(df)):
    img_path = os.path.normpath(row['image_id'])
    split    = row['split']

    # Skip rows not in allowed splits or missing image files
    if split not in ('train','val','test') or not os.path.exists(img_path):
        continue

    images_out_dir = os.path.join(output_root, 'images', split)
    labels_out_dir = os.path.join(output_root, 'labels', split)

    # Copy image to YOLO folder
    img_name = os.path.basename(img_path)
    shutil.copy(img_path, os.path.join(images_out_dir, img_name))

    # Read image size (width, height)
    try:
        w, h = Image.open(img_path).size
    except:
        # If image cannot be opened, skip this row
        continue

    # Convert (xmin, ymin, xmax, ymax) → YOLO (xc, yc, bw, bh), normalized [0,1]
    xmin, ymin, xmax, ymax = (
        row['x_min'], row['y_min'], row['x_max'], row['y_max']
    )
    x_c = (xmin + xmax) / 2 / w
    y_c = (ymin + ymax) / 2 / h
    bw  = (xmax - xmin) / w
    bh  = (ymax - ymin) / h

    # Compose label file path
    label_txt = img_name.rsplit('.', 1)[0] + '.txt'
    with open(os.path.join(labels_out_dir, label_txt), 'a') as f:
        f.write(f"0 {x_c:.6f} {y_c:.6f} {bw:.6f} {bh:.6f}\n")

100%|██████████| 51826/51826 [17:55<00:00, 48.18it/s]  
